# IOAI — 2024 Summer National Needle Haystack (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import numpy as np, torch, os
os.makedirs('data', exist_ok=True)
def _gen(num, nl=10, hl=10, seed=None):
    if seed is not None: np.random.seed(seed)
    npos=num-num//2; X=np.random.randint(2,size=(num,nl+hl))
    needle=np.array((nl//2)*[1.0,0.0])
    for i in range(npos):
        j=np.random.randint(hl); X[i,j:j+nl]=needle
    return torch.tensor(X,dtype=torch.float)[:,:,None]
torch.save(_gen(2000, seed=2024), 'data/needle_test.pt')
print('needle_test.pt regenerated (deterministic, seed=2024)')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Needle in a Haystack — 모범답안 (GRU)

> **HAIO 2024 — Summer National Finals (NLP)**

The vanilla `nn.RNN` baseline can't propagate the alternating-bit pattern across the sequence (~0.5). A **gated RNN (GRU)** carries it easily → ~0.99 detection accuracy. (An LSTM or a 1-D CNN with global max-pool work just as well — see the official sub-tasks 5 & 8.)

In [ ]:
import numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
def get_binary_dataset(num, needle_length=10, haystack_length=10, seed=None):
    # alternating-bit 'needle' inserted at a random position in positive samples;
    # negatives are random bits. First half positive, second half negative.
    if seed is not None: np.random.seed(seed)
    npos = num - num//2
    y = np.array([[1.0]]*npos + [[0.0]]*(num-npos))
    X = np.random.randint(2, size=(num, needle_length+haystack_length))
    needle = np.array((needle_length//2)*[1.0, 0.0])
    for i in range(npos):
        j = np.random.randint(haystack_length); X[i, j:j+needle_length] = needle
    return torch.tensor(X, dtype=torch.float)[:, :, None], torch.tensor(y)
def accuracy(model, X, y):
    model.eval()
    with torch.no_grad(): pred = (model(X) > 0).float()
    return (pred == y).float().mean().item()
def train(model, X, y, epochs=20, lr=1e-2):
    crit = nn.BCEWithLogitsLoss(); opt = torch.optim.Adam(model.parameters(), lr)
    dl = DataLoader(TensorDataset(X, y), batch_size=100, shuffle=True)
    for e in range(epochs):
        model.train()
        for xb, yb in dl: opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, hidden=32, layers=2):
        super().__init__()
        self.rnn = nn.GRU(1, hidden, num_layers=layers, batch_first=True)
        self.fc = nn.Linear(hidden, 1)
    def forward(self, x):
        _, h = self.rnn(x)
        return self.fc(h[-1])

In [ ]:
torch.manual_seed(0)
Xtr, ytr = get_binary_dataset(5000, seed=1)
model = GRUClassifier()
train(model, Xtr, ytr, epochs=20)
Xte = torch.load('data/needle_test.pt')
model.eval()
with torch.no_grad(): pred = (model(Xte) > 0).int().view(-1).tolist()
pd.DataFrame({'id': range(len(pred)), 'label': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(pred), '| predicted positives:', sum(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)